# Module 05: Quality Control with MRIQC

Before investing hours in preprocessing, you must verify that your raw data are
fit for analysis. **MRIQC** is a BIDS App that automatically extracts dozens of
**Image Quality Metrics (IQMs)** and generates visual HTML reports.

## Learning objectives
1. Explain what MRIQC does and why QC is essential.
2. Describe key structural IQMs (CNR, SNR, CJV) and functional IQMs (tSNR, DVARS, FD).
3. Run MRIQC participant-level and group-level via Docker or Singularity.
4. Navigate MRIQC HTML reports.
5. Load group IQM TSVs with pandas, flag outliers, and visualise distributions.

**Prerequisites:** Modules 00-04 completed; Docker ≥ 20.10 or Singularity ≥ 3.7.

**Estimated time:** 2–4 hours.

## 1  Introduction: Why QC Matters

MRI data can be degraded by many factors:

| Problem | Likely cause | Relevant IQM |
|---------|-------------|-------------|
| Signal dropout | Metal implant / susceptibility | SNR, tSNR |
| Motion artefacts | Head movement | FD, DVARS |
| Ghosting | EPI phase errors | GSR |
| Thermal noise | Low field / thin slices | tSNR, CNR |
| Spike artefacts | RF interference | AOR, AQI |

Catching these problems **before preprocessing** saves time and prevents
corrupted subjects from biasing group analyses.

## 2  Key Image Quality Metrics

### Structural (T1w)

| IQM | Full name | Interpretation |
|-----|-----------|----------------|
| `cnr` | Contrast-to-Noise Ratio | Higher = better GM/WM contrast |
| `snr_wm` | SNR in white matter | Higher = less thermal noise |
| `cjv` | Coefficient of Joint Variation | Lower = cleaner GM/WM boundary |
| `efc` | Entropy Focus Criterion | Lower = less ghosting/motion |
| `fber` | Foreground-Background Energy Ratio | Higher = better brain/background |

### Functional (BOLD)

| IQM | Full name | Rule of thumb |
|-----|-----------|---------------|
| `tsnr` | Temporal SNR | ≥ 40 is good; < 20 is concerning |
| `dvars_nstd` | DVARS (normalised) | Spikes > 1.5 may indicate motion |
| `fd_mean` | Mean Framewise Displacement | < 0.5 mm recommended |
| `aor` | AFNI Outlier Ratio | Lower is better |
| `aqi` | AFNI Quality Index | Lower is better |
| `gsr_x/y` | Ghost-to-Signal Ratio | < 0.1 is generally acceptable |

In [ ]:
# Cell 3: Imports and path setup
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from utils.mriqc_helpers import (
    load_group_iqms,
    flag_outliers,
    plot_iqm_distributions,
    generate_exclusion_report,
)

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
print(f'pandas {pd.__version__}  |  numpy {np.__version__}')

## 3  Running MRIQC

### 3a  Docker — participant level

```bash
docker run --rm \
  -v /data/bids:/data:ro \
  -v /data/mriqc:/out \
  -v /data/mriqc/work:/work \
  nipreps/mriqc:23.1.0 \
  /data /out participant \
  --participant-label 01 \
  --work-dir /work \
  --nprocs 4 --mem_gb 8 --no-sub -v
```

### 3b  Singularity — participant level

```bash
singularity run --cleanenv \
  -B /data/bids:/data:ro \
  -B /data/mriqc:/out \
  -B /data/mriqc/work:/work \
  ~/singularity/mriqc_23.1.0.sif \
  /data /out participant \
  --participant-label 01 \
  --work-dir /work \
  --nprocs 4 --mem_gb 8 --no-sub -v
```

### 3c  Convenience scripts

```bash
# Single subject
bash scripts/run_mriqc_single_subject.sh --docker /data/bids /data/mriqc 01

# Group aggregation (after all subjects done)
bash scripts/run_mriqc_group.sh --docker /data/bids /data/mriqc
```

## 4  Group-Level MRIQC

After running participant-level MRIQC for every subject, run the group step:

```bash
docker run --rm \
  -v /data/bids:/data:ro \
  -v /data/mriqc:/out \
  nipreps/mriqc:23.1.0 \
  /data /out group --no-sub
```

This creates:
- `group_bold.tsv` — one row per BOLD run across all subjects
- `group_T1w.tsv`  — one row per T1w scan
- `group_bold.html` / `group_T1w.html` — visual group reports

## 5  Exploring MRIQC HTML Reports

Open `mriqc_output/sub-01/func/sub-01_task-emotionreg_run-1_bold.html` in a browser.

**What to look for:**

| Section | Check |
|---------|-------|
| Summary panel | Overall IQM values; compare to normative ranges |
| Brain mosaic | Coverage, signal dropout, ghosting |
| tSNR map | Uniform signal; low tSNR in frontal/temporal regions indicates susceptibility |
| Carpet plot | Vertical stripes = motion; horizontal stripes = physiological noise |
| FD / DVARS traces | Identify motion spikes for later scrubbing |
| Background noise | Should be smooth and uniform |

In [ ]:
# Cell 6: Simulate a group_bold.tsv for demonstration
# (Replace mriqc_dir with your actual MRIQC output directory)
import tempfile

rng = np.random.default_rng(42)
n_runs = 20
subjects = [f'sub-{i:02d}' for i in range(1, n_runs + 1)]

simulated = pd.DataFrame({
    'bids_name': subjects,
    'subject':   subjects,
    'tsnr':      rng.normal(55, 10, n_runs),
    'dvars_nstd':rng.normal(1.2, 0.3, n_runs),
    'fd_mean':   rng.exponential(0.25, n_runs),
    'aor':       rng.normal(0.02, 0.005, n_runs),
    'aqi':       rng.normal(0.005, 0.001, n_runs),
    'gsr_x':     rng.normal(0.05, 0.02, n_runs),
    'snr':       rng.normal(8, 1.5, n_runs),
})

# Inject two outliers
simulated.loc[3, 'tsnr'] = 15.0
simulated.loc[11, 'fd_mean'] = 1.8

# Write to a temp dir (mimics MRIQC output)
MRIQC_DIR = tempfile.mkdtemp(prefix='mriqc_demo_')
simulated.to_csv(os.path.join(MRIQC_DIR, 'group_bold.tsv'), sep='\t', index=False)

print(f'Demo MRIQC dir: {MRIQC_DIR}')
simulated.head()

In [ ]:
# Cell 7: Load group IQMs using utils.mriqc_helpers
iqms = load_group_iqms(MRIQC_DIR, modality='bold')

print(f'Loaded {len(iqms)} runs, {len(iqms.columns)} columns.')
print('\nFirst 5 rows:')
iqms.head()

In [ ]:
# Cell 8: Descriptive statistics of key IQMs
key_metrics = ['tsnr', 'dvars_nstd', 'fd_mean', 'aor', 'gsr_x', 'snr']
available = [m for m in key_metrics if m in iqms.columns]

print('Key IQM summary statistics:')
iqms[available].describe().round(3)

In [ ]:
# Cell 9: Flag outliers using IQR method
flags = flag_outliers(iqms, metrics=available, threshold=2.5)

n_flagged = int(flags['any_outlier'].sum())
print(f'Flagged {n_flagged} / {len(iqms)} runs as outliers.\n')

if 'bids_name' in iqms.columns:
    flagged_ids = iqms.loc[flags['any_outlier'], 'bids_name'].tolist()
    print('Flagged runs:', flagged_ids)

print('\nOutlier flag table (True = outlier):')
flags[flags['any_outlier']]

In [ ]:
# Cell 10: Visualise IQM distributions
fig = plot_iqm_distributions(iqms, metrics=available)

In [ ]:
# Cell 11: tSNR vs mean FD scatter — spot motion-quality trade-off
if 'tsnr' in iqms.columns and 'fd_mean' in iqms.columns:
    fig, ax = plt.subplots(figsize=(6, 4))
    colors = ['red' if f else 'steelblue' for f in flags['any_outlier']]
    ax.scatter(iqms['fd_mean'], iqms['tsnr'], c=colors, s=60, alpha=0.8, edgecolors='k', linewidths=0.5)
    ax.set_xlabel('Mean Framewise Displacement (mm)')
    ax.set_ylabel('tSNR')
    ax.set_title('tSNR vs Mean FD\n(red = IQR outlier)')
    ax.axhline(40, color='orange', linestyle='--', linewidth=1, label='tSNR = 40')
    ax.axvline(0.5, color='purple', linestyle='--', linewidth=1, label='FD = 0.5 mm')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 12: Generate exclusion report
report = generate_exclusion_report(iqms)

In [ ]:
# Cell 13: Run full analysis script programmatically
scripts_dir = os.path.join(os.getcwd(), 'scripts')
sys.path.insert(0, scripts_dir)

from analyze_mriqc_output import main as analyze_main

out_dir = os.path.join(MRIQC_DIR, 'qc_figures')
exit_code = analyze_main([
    '--mriqc_dir',  MRIQC_DIR,
    '--output_dir', out_dir,
    '--modality',   'bold',
])
print(f'\nScript exit code: {exit_code}  (0 = success)')
print('Output files:')
for f in os.listdir(out_dir):
    print(f'  {f}')

## 6  Summary and Next Steps

In this module you learned how to:

- Explain what MRIQC measures and why it matters.
- Interpret key structural IQMs (CNR, SNR, CJV) and functional IQMs (tSNR, FD, DVARS).
- Run MRIQC participant-level and group-level via Docker or Singularity.
- Navigate individual and group HTML reports.
- Load group IQM TSV files with `utils.mriqc_helpers.load_group_iqms()`.
- Flag outlier scans using IQR-based statistics.
- Visualise IQM distributions and produce an exclusion report.

### Next

➡️  **Module 06: fMRIPrep** — preprocess your quality-checked BIDS dataset.

### References

- [MRIQC documentation](https://mriqc.readthedocs.io/)
- [Esteban et al. (2017) MRIQC paper](https://doi.org/10.1371/journal.pone.0184661)
- [MRIQC IQM reference](https://mriqc.readthedocs.io/en/latest/measures.html)
- [fMRIPrep documentation](https://fmriprep.org/)